# 🎬 Channel Micro-POV — 1-Click Cloud Render Engine (Google Colab + Blender 5.2.2)

**Prinsip Zero-Manual & Satu Pintu:** 100% proses jalan di cloud tanpa laptop kencang.
- **Karakter Manusia Asli MPFB2** (`family_ready.blend` otomatis disalin dari Google Drive ke cache proyek `02_assets/models_3d/`)
- **15 Properti Dapur Nyata Poly Haven CC0** + tekstur PBR keramik & plester kosan + HDRI
- **Lighting Jam 2 Pagi + AgX + Macro DOF + `post.py`** (soft lens, bloom glow, vignette, film grain)
- **Audio-First Gemini 3.8 Flash TTS (`Puck`)** dengan cache Drive (`SKIP_EXISTING_VO=True`, 24fps, `VO_SPEED=1.08`)
- **Auto-Resume per Frame** (> 10 KB dilewati) + pilihan engine **`CYCLES`** (OptiX/CUDA + OIDN) atau **`EEVEE`** (cepat 2–6 dtk/frame)

### Pilihan Mode (`RENDER_MODE`):
- **`storyboard` (Cepat — jalan di CPU/GPU):** 8 hero frame (`360×640`) + `post.py` → `storyboard_grid.png` + `animatic_epXXX.mp4`.
- **`preview` (T4 GPU):** Full 24fps `540×960` untuk QC gerak kamera & akting MPFB2.
- **`final` (T4 GPU):** Full 24fps `1080×1920` siap upload YouTube Shorts / TikTok / Reels.

In [ ]:
#@title 1. Konfigurasi Episode, Engine & Mount Google Drive { run: "auto" }
import os
import shutil
from pathlib import Path

REPO_URL = "https://github.com/ruangcermat/channel-micro-pov.git" #@param {type:"string"}
REPO_BRANCH = "main" #@param {type:"string"}
EPISODE_SPEC = "04_episodes/ep001_kecoak_dapur/ep001_spec.json" #@param {type:"string"}
RENDER_MODE = "storyboard" #@param ["storyboard", "preview", "final"]
RENDER_ENGINE = "CYCLES" #@param ["CYCLES", "EEVEE"]
SINGLE_SCENE = 0 #@param {type:"integer"}
TTS_VOICE = "Puck" #@param ["Puck", "Charon", "Fenrir", "Kore", "Aoede", "Enceladus"]
VO_SPEED = 1.08 #@param {type:"number"}
SKIP_EXISTING_VO = True #@param {type:"boolean"}
SAVE_PACKED_BLEND = False #@param {type:"boolean"}

# 1. Mount Google Drive untuk cache aset & penyimpanan output permanen
from google.colab import drive, userdata
drive.mount("/content/drive", force_remount=False)

DRIVE_ROOT = Path("/content/drive/MyDrive/channel-micro-pov")
ASSETS_DIR = DRIVE_ROOT / "02_assets"
EPISODE_SLUG = Path(EPISODE_SPEC).parent.name
WORK_DIR = DRIVE_ROOT / "04_episodes" / EPISODE_SLUG

ASSETS_DIR.mkdir(parents=True, exist_ok=True)
(ASSETS_DIR / "models_3d").mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)

# 2. Auto-sync MPFB2 family_ready.blend dari Google Drive ke cache proyek
FAMILY_CACHE = ASSETS_DIR / "models_3d" / "family_ready.blend"
FAMILY_DRIVE_SOURCES = [
    Path("/content/drive/MyDrive/Skill referensi/Zack Blender Pack/Zack-Blender-Pack/zack-3d-short/ready/family_ready.blend"),
    DRIVE_ROOT / "05_skills_ref/Zack-Blender-Pack/zack-3d-short/ready/family_ready.blend",
]
if not FAMILY_CACHE.exists():
    for src in FAMILY_DRIVE_SOURCES:
        if src.exists():
            print(f"📦 Menyalin MPFB2 family_ready.blend ke cache proyek: {FAMILY_CACHE}")
            shutil.copy2(src, FAMILY_CACHE)
            break
if FAMILY_CACHE.exists():
    os.environ["MICRO_POV_FAMILY_BLEND"] = str(FAMILY_CACHE)
    print(f"✅ MPFB2 family_ready.blend siap ({FAMILY_CACHE.stat().st_size / 1e6:.1f} MB)")
else:
    print("⚠️ family_ready.blend tidak ditemukan di Drive! Pastikan folder Zack-Blender-Pack ada di Google Drive.")

# 3. Baca Secrets opsional (GEMINI_API_KEY & GITHUB_TOKEN jika repo private)
GITHUB_TOKEN = ""
for secret_name in ("GEMINI_API_KEY", "GITHUB_TOKEN"):
    try:
        val = userdata.get(secret_name)
        if val and val.strip():
            if secret_name == "GITHUB_TOKEN":
                GITHUB_TOKEN = val.strip()
            else:
                os.environ[secret_name] = val.strip()
            print(f"✅ Secret {secret_name} dimuat dari Colab Secrets.")
    except Exception:
        pass

os.environ["MICRO_POV_ASSETS_DIR"] = str(ASSETS_DIR)
os.environ["MICRO_POV_WORK_DIR"] = str(WORK_DIR)

print(f"📁 Drive Assets Cache : {ASSETS_DIR}")
print(f"📁 Episode Output Dir : {WORK_DIR}")
print(f"🎞️ Mode: {RENDER_MODE} | Engine: {RENDER_ENGINE} | Scene: {SINGLE_SCENE or 'ALL'} | VO Speed: {VO_SPEED}x | 24fps")
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader || echo "ℹ️ Mode CPU aktif (cocok untuk storyboard; gunakan T4 GPU untuk preview/final)."

In [ ]:
#@title 2. Sync Repository GitHub & Install Dependensi Pipeline
import subprocess

def run_step(cmd):
    """Jalankan subprocess dengan output penuh real-time (stdout + stderr)."""
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="", flush=True)
    rc = proc.wait()
    if rc != 0:
        raise subprocess.CalledProcessError(rc, cmd)

AUTH_REPO_URL = REPO_URL
if GITHUB_TOKEN and REPO_URL.startswith("https://"):
    AUTH_REPO_URL = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@", 1)

REPO_DIR = Path("/content/channel-micro-pov")
if (REPO_DIR / ".git").exists():
    print("🔄 Pulling update terbaru dari GitHub...")
    run_step(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", AUTH_REPO_URL])
    run_step(["git", "-C", str(REPO_DIR), "fetch", "origin"])
    run_step(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH])
    run_step(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", REPO_BRANCH])
else:
    print("📥 Cloning repository dari GitHub...")
    run_step(["git", "clone", "-b", REPO_BRANCH, AUTH_REPO_URL, str(REPO_DIR)])

run_step(["bash", "-lc", "apt-get update -qq && apt-get install -y -qq ffmpeg fonts-dejavu-core fonts-liberation libxrender1 libxxf86vm1 libxfixes3 libxi6 libxkbcommon0 libsm6 libgl1 libegl1 libglu1-mesa > /dev/null"])
run_step(["pip", "install", "-q", "--upgrade", "-r", str(REPO_DIR / "03_pipeline/requirements.txt")])
print("✅ Repo & dependensi Python/FFmpeg/Font/GL siap!")

In [ ]:
#@title 3. Install Blender 5.2.2 LTS Linux Headless (Sinkron dengan family_ready.blend & MPFB2)
BLENDER_VERSION = "5.2.2"
BLENDER_TAR = f"blender-{BLENDER_VERSION}-linux-x64.tar.xz"
BLENDER_URL = f"https://download.blender.org/release/Blender5.2/{BLENDER_TAR}"
BLENDER_BIN = Path("/opt/blender52/blender")

if not BLENDER_BIN.exists():
    print(f"⬇️ Mengunduh Blender {BLENDER_VERSION} LTS Linux Headless...")
    run_step(["wget", "-q", "--show-progress", BLENDER_URL, "-O", f"/tmp/{BLENDER_TAR}"])
    run_step(["mkdir", "-p", "/opt/blender52"])
    run_step(["tar", "-xf", f"/tmp/{BLENDER_TAR}", "-C", "/opt/blender52", "--strip-components=1"])
    run_step(["rm", "-f", f"/tmp/{BLENDER_TAR}"])
else:
    print("✅ Blender 5.2.2 headless sudah terpasang di /opt/blender52")

run_step([str(BLENDER_BIN), "--version"])

In [ ]:
#@title 4. Auto-Fetch 15 Properti Poly Haven CC0 + Tekstur PBR + Sync MPFB2 Cache
SPEC_PATH = REPO_DIR / EPISODE_SPEC
run_step([
    "python3", str(REPO_DIR / "03_pipeline/scripts/fetch_assets.py"),
    "--spec", str(SPEC_PATH),
    "--assets-dir", str(ASSETS_DIR),
])

In [ ]:
#@title 5. Audio-First Voiceover — Gemini 3.8 Flash TTS (Cache Drive + 24fps + VO Speed)
import json
from IPython.display import Audio, display

tts_cmd = [
    "python3",
    str(REPO_DIR / "03_pipeline/scripts/generate_tts.py"),
    "--spec", str(SPEC_PATH),
    "--work-dir", str(WORK_DIR),
    "--voice", TTS_VOICE,
    "--fps", "24",
    "--pad-sec", "0.06",
    "--vo-speed", str(VO_SPEED),
]
if SKIP_EXISTING_VO:
    tts_cmd.append("--skip-existing")
if SINGLE_SCENE > 0:
    tts_cmd.extend(["--scene", str(SINGLE_SCENE)])

run_step(tts_cmd)

manifest_path = WORK_DIR / "timing_manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
print(f"\n⏱️ Total durasi Audio-First ({manifest.get('vo_speed', 1.0)}x): {manifest['total_duration_sec']:.2f}s ({manifest['total_frames']} frames @ {manifest['fps']}fps)")
for sc in manifest["scenes"]:
    print(
        f"  Scene {sc['scene_id']:02d}: {sc['start_sec']:5.2f}s - {sc['end_sec']:5.2f}s "
        f"(VO {sc['audio_duration_sec']:.2f}s, {sc['frame_count']}f) | {sc['vo_text'][:52]}..."
    )

first_wav = Path(manifest["scenes"][0]["wav_path"])
if first_wav.exists():
    print("\n🔊 Preview VO Scene 01:")
    display(Audio(filename=str(first_wav)))

In [ ]:
#@title 6. Auto-Prepare SFX & BGM CC0
run_step([
    "python3", str(REPO_DIR / "03_pipeline/scripts/prepare_audio.py"),
    "--spec", str(SPEC_PATH),
    "--assets-dir", str(ASSETS_DIR),
])

In [ ]:
#@title 7. Blender 5.2.2 Render (MPFB2 + 15 CC0 Props + AgX + Macro DOF + post.py + Auto-Resume)
from IPython.display import Image as IPyImage, display

render_cmd = [
    str(BLENDER_BIN),
    "--background",
    "--python", str(REPO_DIR / "03_pipeline/scripts/build_and_render.py"),
    "--",
    "--spec", str(SPEC_PATH),
    "--timing", str(WORK_DIR / "timing_manifest.json"),
    "--assets-dir", str(ASSETS_DIR),
    "--work-dir", str(WORK_DIR),
    "--mode", RENDER_MODE,
    "--engine", RENDER_ENGINE,
]
if SAVE_PACKED_BLEND:
    render_cmd.append("--save-blend")
if SINGLE_SCENE > 0:
    render_cmd.extend(["--scene", str(SINGLE_SCENE)])

print("🚀 Menjalankan Blender 5.2.2:", " ".join(render_cmd))
run_step(render_cmd)

sheet_path = WORK_DIR / "storyboard_sheet.png"
grid_path = WORK_DIR / "storyboard_grid.png"
if RENDER_MODE == "storyboard" and sheet_path.exists():
    print(f"\n🎬 Studio Storyboard Sheet 9 Panel (lengkap dengan kartu aksi/VO/SFX + post.py): {sheet_path}")
    display(IPyImage(filename=str(sheet_path)))
elif RENDER_MODE == "storyboard" and grid_path.exists():
    print(f"\n🖼️ Storyboard Contact Sheet 2×4 (lengkap dengan post.py): {grid_path}")
    display(IPyImage(filename=str(grid_path)))

In [ ]:
#@title 8. Subtitle Dinamis (.ass 9:16 Anti-Terpotong) & Mixing Final (-14 LUFS + Auto-Ducking)
run_step([
    "python3", str(REPO_DIR / "03_pipeline/scripts/compose_final.py"),
    "--spec", str(SPEC_PATH),
    "--timing", str(WORK_DIR / "timing_manifest.json"),
    "--assets-dir", str(ASSETS_DIR),
    "--work-dir", str(WORK_DIR),
    "--mode", RENDER_MODE,
])

In [ ]:
#@title 9. Player Preview Video Langsung di Colab & Lokasi File Google Drive
import base64
from IPython.display import HTML, display

ep_id = manifest["episode_id"].lower()
if RENDER_MODE == "final":
    out_name = f"final_{ep_id}.mp4"
elif RENDER_MODE == "storyboard":
    out_name = f"animatic_{ep_id}.mp4"
else:
    out_name = f"preview_{ep_id}.mp4"
video_path = WORK_DIR / out_name

assert video_path.exists(), f"File video tidak ditemukan: {video_path}"
size_mb = video_path.stat().st_size / (1024 * 1024)
print(f"🎉 SELESAI! Video tersimpan permanen di Google Drive:")
print(f"   📂 {video_path} ({size_mb:.2f} MB)")

if size_mb <= 35.0:
    b64 = base64.b64encode(video_path.read_bytes()).decode("ascii")
    html_player = (
        '<div style="display:flex;justify-content:center;padding:12px;background:#111;border-radius:12px;">'
        '<video width="360" height="640" controls playsinline '
        'style="border-radius:10px;box-shadow:0 8px 24px rgba(0,0,0,0.5);">'
        f'<source src="data:video/mp4;base64,{b64}" type="video/mp4">'
        "</video></div>"
    )
    display(HTML(html_player))
else:
    print("ℹ️ Ukuran file > 35 MB; silakan putar langsung dari Google Drive kamu.")